# Lab I — Inference Optimization on the free tier

**Curriculum §5 · Track 2**

The full track serves with vLLM on a real GPU (batching, KV-cache, quantization, speculative decoding). Those need a dedicated box. Here we optimize the levers you can measure **free** — **caching, routing, and context trimming** — and cost them, the same way the [cost-optimization guide](https://careerstack.dev/ai-infra-cost-optimization) lays out.

> Runs against Groq's free API; the *techniques* transfer 1:1 to self-hosted serving.

## ▶ Colab setup — run this cell first

1. Add your keys in Colab: **🔑 (left sidebar) → Secrets** → add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `GROQ_API_KEY` (toggle *Notebook access* on).
2. Free signups: **Langfuse** → cloud.langfuse.com · **Groq** → console.groq.com
3. Run the cell. It installs deps and clones the shared `common/` package from GitHub.

> **If you see a `PIL._typing._Ink` import error:** run the cell, then **Runtime → Restart session**, then re-run. It's a Colab package clash, fixed by the Pillow upgrade + a restart.

In [ ]:
# --- Colab bootstrap (safe to re-run) ---
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run('pip install -q -U Pillow'.split())          # fix Colab PIL._typing._Ink clash
    subprocess.run('pip install -q langfuse ragas sentence-transformers faiss-cpu rank_bm25 langchain langchain-community langchain-groq langchain-text-splitters langgraph pypdf pdfplumber PyMuPDF pandas'.split())

    # The repo is public — clone it to get the shared common/ package.
    REPO = pathlib.Path('/content/labpractice')
    if not (REPO/'common'/'harness.py').exists():
        subprocess.run(['git','clone','--depth','1',
                        'https://github.com/AICareerstack26/labpractice', str(REPO)])
    sys.path.insert(0, str(REPO))

    # Keys from Colab Secrets (🔑 sidebar) -> env vars that common/config.py reads.
    try:
        from google.colab import userdata
        for k in ['LANGFUSE_PUBLIC_KEY','LANGFUSE_SECRET_KEY','GROQ_API_KEY']:
            v = userdata.get(k)
            if v: os.environ[k] = v
        os.environ.setdefault('LANGFUSE_HOST','https://cloud.langfuse.com')
    except Exception as e:
        print('Secrets not set — running offline. (', e, ')')
else:
    sys.path.insert(0, str(pathlib.Path.cwd().parent))          # local fallback

print('Environment:', 'Colab' if IN_COLAB else 'Local')

In [ ]:
import pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd().parent))              # make `common` importable locally
from common.config import *
from common.corpus import DOCS, current_docs, SUPERSEDED_IDS
from common.golden import GOLDEN
from common.obs import observe, span_meta, trace_meta, make_config, flush, ENABLED
from common.harness import evaluate, leaderboard
print('Langfuse tracing:', 'ON' if ENABLED else 'OFF (labs still run)')
print('AS_OF:', AS_OF, '| in force:', [d['id'] for d in current_docs()], '| superseded:', SUPERSEDED_IDS)

## 1 · Baseline — measure before you optimize

You cannot claim a speed-up without the floor. We reuse the version-safe RAG and time every golden query.

In [ ]:
import numpy as np, time
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

split = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
CHUNKS = [dict(id=d['id'], text=f"[{d['doc']} {d['version']}] {p}")
          for d in current_docs() for p in split.split_text(d['text'])]
M_emb = SentenceTransformer(EMBED_MODELS['base'])
MAT   = M_emb.encode([c['text'] for c in CHUNKS], normalize_embeddings=True)

big   = ChatGroq(model=GEN_MODEL,  temperature=0)   # llama-3.3-70b
small = ChatGroq(model=SMALL_MODEL, temperature=0)  # llama-3.1-8b-instant
PRICE = {GEN_MODEL: 0.59, SMALL_MODEL: 0.05}        # ~ $/1M tokens (illustrative)

def retrieve(query, k=3):
    qv = M_emb.encode([query], normalize_embeddings=True)[0]
    idx = np.argsort(-(MAT @ qv))[:k]
    return [dict(CHUNKS[i]) for i in idx]

PROMPT = ("You are Meridian Bank's policy copilot. Answer ONLY from context, cite ids in [brackets], "
          "else INSUFFICIENT_CONTEXT.\n\nContext:\n{ctx}\n\nQuestion: {q}")
def est_tokens(s): return max(1, len(s)//4)          # cheap token proxy

## 2 · Four optimizations, each measured

`exact cache` · `semantic cache` · `model routing` (easy→8B, hard→70B) · `context trim` (fewer chunks).

In [ ]:
EXACT = {}                                            # exact-match cache
SEM_Q, SEM_A = [], []                                  # semantic cache stores (vec, answer)

def answer(query, mode='baseline'):
    t0 = time.time(); toks = 0; model = GEN_MODEL
    if mode in ('exact','all') and query in EXACT:
        return EXACT[query], time.time()-t0, 0, 'cache-exact'
    if mode in ('semantic','all') and SEM_Q:
        qv = M_emb.encode([query], normalize_embeddings=True)[0]
        sims = M_emb.encode(SEM_Q, normalize_embeddings=True) @ qv if SEM_Q else []
        if len(sims) and float(np.max(sims)) > 0.97:
            return SEM_A[int(np.argmax(sims))], time.time()-t0, 0, 'cache-semantic'

    k = 2 if mode in ('trim','all') else 3
    hits = retrieve(query, k=k)
    ctx  = '\n'.join(f"[{h['id']}] {h['text']}" for h in hits)
    p    = PROMPT.format(ctx=ctx, q=query)

    llm = big
    if mode in ('route','all'):                        # route short/lookup queries to the 8B
        if len(query.split()) <= 12: llm, model = small, SMALL_MODEL
    ans = llm.invoke(p).content
    toks = est_tokens(p) + est_tokens(ans)
    EXACT[query] = ans
    SEM_Q.append(query); SEM_A.append(ans)
    return ans, time.time()-t0, toks*PRICE[model]/1e6, model

## 3 · Run each mode over the golden set (twice — so caches can hit)

In [ ]:
import pandas as pd
rows = []
for mode in ['baseline','exact','semantic','route','trim','all']:
    EXACT.clear(); SEM_Q.clear(); SEM_A.clear()
    lat, cost = [], []
    for _round in range(2):                            # 2nd pass exercises the caches
        for g in GOLDEN:
            _, dt, c, _ = answer(g['q'], mode=mode)
            lat.append(dt); cost.append(c)
    rows.append(dict(mode=mode, p50_latency_s=round(float(np.percentile(lat,50)),3),
                     p95_latency_s=round(float(np.percentile(lat,95)),3),
                     cost_per_100=round(sum(cost)/len(cost)*100,4)))
pd.DataFrame(rows)

## 4 · What you should conclude

- **Caching** is the biggest free win — a warm exact/semantic cache drops repeat latency and cost to ~0.
- **Routing** short lookups to the 8B model cuts blended $/query with no quality loss *on those queries* — measure the misroute rate on your eval set before trusting it.
- **Context trimming** shaves tokens; past a point it costs `hit_at_k`, so keep it on the harness's leash.

> The levers here are the *free* half of serving. The other half — batching, KV/prefix cache, quantization, speculative decoding — needs vLLM on a real GPU, but the discipline is identical: **measure p95 and $/query, change one lever, re-measure.**

**Next →** `lab05` (fine-tuning)